In [ ]:
import numpy as np
from activators import ReluActivator,IdentityActivator

# **辅助函数**

In [ ]:
def get_patch(input_array,i,j,filter_width,filter_height,stride):
    '''
    :param input_array: 输入数组
    :param i: 图像上的第i行
    :param j: 图像上的第j列
    :param filter_width: 卷积核的宽
    :param filter_height: 卷积核的高
    :param stride: 步长
    :return: 切出的方块
    '''
    # 小方块左上角的坐标
    start_i=i*stride
    start_j=j*stride
    # 如果是单通道图片：
    if input_array.ndim==2:
        return input_array[start_i:start_i+filter_height,start_j:start_j+filter_width]
#     如果是多通道图片：
    elif input_array.ndim==3:
        return input_array[:,start_i : start_i + filter_height,start_j : start_j + filter_width]
def get_max_index(array):
    '''
    输入一个二维数组，用于最大池化，找最大值位置并返回坐标
    '''
    max_i=0
    max_j=0
    max_value=array[0,0]

    for i in range(array.shape[0]):
        for j in range(array.shape[1]):
            if array[i,j]>max_value:
                max_value=array[i,j]
                max_i=i
                max_j=j
    return max_i,max_j
def conv(input_array,kernel_array,output_array,stride,bias):
    '''
    :param input_array: 做好padding的输入数组
    :param kernel_array:卷积核权重
    :param output_array:输出数组
    :param stride:步长
    :param bias:偏置
    '''
    channel_number=input_array.ndim
    output_width=output_array.shape[1]
    output_height=output_array.shape[0]

    kernel_width=kernel_array.shape[-1]
    kernel_height=kernel_array.shape[-2]

    for i in range(output_height):
        for j in range(output_width):
            patch=get_patch(input_array,i,j,kernel_width,kernel_height,stride)
            sum_value=(patch*kernel_array).sum()
            output_array[i][j]=sum_value+bias
def padding(input_array,zp):
    '''
    给数组四周补0，zp是圈数，返回补充后的数组，补充后宽高分别增加2zp
    '''
    if zp==0:
        return input_array
    if input_array.ndim==3:
        input_depth=input_array.shape[0]
        input_height=input_array.shape[1]
        input_width=input_array.shape[2]

        padded_array=np.zeros([input_depth,input_height+2*zp,input_width+2*zp])
        padded_array[:,zp:zp+input_height,zp:zp+input_width]=input_array
        return padded_array
    elif input_array.ndim==2:
        input_height=input_array.shape[0]
        input_width=input_array.shape[1]
        padded_array=np.zeros([input_height+2*zp,input_width+2*zp])
        padded_array[zp:zp+input_height,zp:zp+input_width]=input_array
        return padded_array
def element_wise_op(array,op):
    '''
    :param op: 操作
    对数组每个元素应用op函数
    '''
    # np.nditer是迭代器，op_flags=['readwrite']表示可以读取修改元素，i[...]是修改元素的标准写法
    for i in np.nditer(array,op_flags=['readwrite']):
        i[...]=op(i)


# **卷积核类**

In [ ]:
# 卷积核包含权重(权重形状(depth, height, width))和偏置
class Filter(object):
    def __init__(self,width,height,depth):
        self.weights=np.random.uniform(-1e-4,1e-4,(depth,height,width))
        self.bias=0
        self.weights_grad=np.zeros(self.weights.shape)
        self.bias_grad=0
    def update(self,learning_rate):
        '''
        W_new = W_old - η × (∂E/∂W)
        b_new = b_old - η × (∂E/∂b)
        '''
        self.weights-=learning_rate * self.weights_grad
        self.bias-=learning_rate * self.bias_grad




